<a href="https://colab.research.google.com/github/thomaslu678/Praxis-Lab-25-26/blob/main/clean/18_Similarity_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
pip install kgcPy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 834.6/834.6 kB 9.1 MB/s eta 0:00:00


# Get feature weight vector

In [71]:
import io
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [72]:
raw_csvs = []
user_weights = [0.25, 0.25, 0.5]

In [73]:
np.sum(user_weights)

np.float64(1.0)

In [74]:
user_1 = pd.read_csv(io.StringIO('''
,1/6,1/5,1/3,1/8
,,2,5,1/2
,,,3,1/4
,,,,1/8
,,,,
'''), header=None)

In [75]:
# 2. Helper function to turn fraction strings into actual floats
def parse_fraction(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    val = str(val).strip()
    if "/" in val:
        num, denom = val.split("/")
        return float(num) / float(denom)
    return float(val)

def get_numeric_df(sheets_csv):
    df_numeric = sheets_csv.map(parse_fraction)

    # 3. Populate the main diagonal with 1s
    np.fill_diagonal(df_numeric.values, 1.0)

    # 4. Fill the lower triangle with the reciprocals of the upper triangle
    # We iterate only through the lower triangle coordinates
    for i in range(len(df_numeric)):
        for j in range(i):
            df_numeric.iloc[i, j] = 1.0 / df_numeric.iloc[j, i]

    return df_numeric

In [76]:
def get_real_principal_eigenvector(df_numeric):
    # Compute eigenvalues and right eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(df_numeric)

    # Find the index of the largest eigenvalue
    max_index = np.argmax(eigenvalues)

    # Extract the principal eigenvector (stored as a column)
    principal_eigenvector = eigenvectors[:, max_index]

    return principal_eigenvector.real

In [77]:
raw_csvs.append(user_1)
raw_csvs.append(user_1)
raw_csvs.append(user_1)

In [78]:
raw_weights = []

In [79]:
for csv in raw_csvs:
    weight = get_real_principal_eigenvector(get_numeric_df(csv))
    sum = np.sum(weight)
    normalized = weight / sum
    raw_weights.append(normalized)

In [80]:
raw_weights

[array([0.0376873 , 0.26562517, 0.15387367, 0.06659161, 0.47622226]),
 array([0.0376873 , 0.26562517, 0.15387367, 0.06659161, 0.47622226]),
 array([0.0376873 , 0.26562517, 0.15387367, 0.06659161, 0.47622226])]

In [81]:
def edbam(individual_weights, stakeholder_weights):
    """
    Calculate the Euclidean Distance-Based Aggregation Method (EDBAM)
    group weight vector.

    Parameters
    ----------
    individual_weights : np.ndarray
        Array of shape (m, n), where each row is one stakeholder's
        normalized principal eigenvector.

    stakeholder_weights : np.ndarray
        Array of shape (m,), containing the stakeholder weights.
        Must sum to 1.

    Returns
    -------
    group_weights : np.ndarray
        EDBAM aggregated weight vector of shape (n,).
        Sums to 1.
    """

    individual_weights = np.asarray(individual_weights, dtype=float)
    stakeholder_weights = np.asarray(stakeholder_weights, dtype=float)

    # Basic validation
    if individual_weights.ndim != 2:
        raise ValueError("individual_weights must be a 2D array.")

    m, n = individual_weights.shape

    if stakeholder_weights.shape != (m,):
        raise ValueError(
            "stakeholder_weights must have one value per stakeholder."
        )

    if np.any(individual_weights < 0):
        raise ValueError("Individual weight vectors must be non-negative.")

    if np.any(stakeholder_weights < 0):
        raise ValueError("Stakeholder weights must be non-negative.")

    if not np.isclose(stakeholder_weights.sum(), 1.0):
        raise ValueError("Stakeholder weights must sum to 1.")

    if not np.allclose(individual_weights.sum(axis=1), 1.0):
        raise ValueError(
            "Each individual weight vector must sum to 1."
        )

    # Objective function:
    #
    # f(x) = sum_k a_k * ||w^(k) - x||_2
    #
    def objective(x):
        distances = np.linalg.norm(individual_weights - x, axis=1)
        return np.dot(stakeholder_weights, distances)

    # Initial guess: weighted arithmetic mean.
    # This is a convenient starting point, NOT the EDBAM solution.
    x0 = np.average(
        individual_weights,
        axis=0,
        weights=stakeholder_weights
    )

    # Constrain x to be a valid weight vector:
    # x_i >= 0
    # sum(x_i) = 1
    constraints = {
        "type": "eq",
        "fun": lambda x: np.sum(x) - 1.0
    }

    bounds = [(0.0, 1.0)] * n

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "ftol": 1e-12,
            "maxiter": 1000
        }
    )

    if not result.success:
        raise RuntimeError(
            f"EDBAM optimization failed: {result.message}"
        )

    # Final normalization for numerical precision
    group_weights = result.x / result.x.sum()

    return group_weights

In [82]:
group_weights = edbam(
    raw_weights,
    user_weights
)

print(group_weights)
print(group_weights.sum())

[0.0376873  0.26562517 0.15387367 0.06659161 0.47622226]
1.0000000000000002


# Get all similarity scores

In [174]:
data = pd.read_csv(io.StringIO('''
ID,xmin,ymin,Duration,Pop. Start
0,,,7,"307,006"
1,-84.37347649,33.75400957,3,"483,450"
2,,,2,"472,330"
3,-87.7126679,41.91046952,2,"2,718,782"
4,-99.20962876,19.43531808,1,"8,918,653"
5,-99.11203944,19.43137642,3,"8,920,000"
6,-111.9105146,40.74596957,6,"200,546"
7,127.0737116,37.61597727,9,"2,660,000"
8,126.9129689,37.5548156,6,"2,660,000"
9,-74.00025074,40.7496983,8,"8,214,426"
10,2.253137216,48.91112189,11,"2,120,545"
11,-84.51188626,38.03708055,4,"322,600"
12,-87.95937097,43.03146501,5,"604,477"
13,-93.32778934,44.94513612,7,"368,380"
14,-46.54968154,-23.51900561,4,"17,962,000"
15,2.367702968,48.8369321,12,"2,158,732"
16,-71.06480015,42.34998438,17,"574,282"
17,126.9632178,37.55166639,2,"2,890,000"
18,12.53197094,55.65926482,16,"502,672"
19,12.53615059,55.69788659,2,"536,743"
20,151.1965143,-33.8845652,3,"189,543"
21,-102.2601346,21.84015311,2,"1,215,094"
22,2.125247295,41.36614561,1,"1,610,427"
'''), header=None)

In [175]:
data = data.drop(1)
data = data.drop(3)

In [176]:
data.columns = data.iloc[0]
data = data[1:]

In [177]:
data['Climate'] = None

In [178]:
data = data.reset_index(drop=True)

In [183]:
import pandas as pd
import rasterio
from rasterio.windows import Window
from urllib.request import urlopen


# -------------------------------------------------------------------
# 1. Köppen-Geiger raster
# -------------------------------------------------------------------

RASTER_URL = (
    "https://data.naturalcapitalalliance.stanford.edu/download/global/koppen_geiger_climatezones/koppen_geiger_climatezones_1991_2020_1km.tif"
)


# -------------------------------------------------------------------
# 2. Köppen-Geiger class lookup
#
# The Beck/Köppen-Geiger raster stores classes as integer values.
# These are the standard 30 Köppen-Geiger sub-classes.
# -------------------------------------------------------------------

KOPPEN_CLASSES = {
    1:  "Af",
    2:  "Am",
    3:  "Aw",
    4:  "BWh",
    5:  "BWk",
    6:  "BSh",
    7:  "BSk",
    8:  "Csa",
    9:  "Csb",
    10: "Csc",
    11: "Cwa",
    12: "Cwb",
    13: "Cwc",
    14: "Cfa",
    15: "Cfb",
    16: "Cfc",
    17: "Dsa",
    18: "Dsb",
    19: "Dsc",
    20: "Dsd",
    21: "Dwa",
    22: "Dwb",
    23: "Dwc",
    24: "Dwd",
    25: "Dfa",
    26: "Dfb",
    27: "Dfc",
    28: "Dfd",
    29: "ET",
    30: "EF",
}


# -------------------------------------------------------------------
# 3. Function to classify coordinates
# -------------------------------------------------------------------

def get_koppen_zones(lons, lats):
    """
    Convert longitude/latitude coordinates to Köppen-Geiger
    climate classifications.

    Parameters
    ----------
    lons : array-like
        Longitude in decimal degrees, WGS84.
    lats : array-like
        Latitude in decimal degrees, WGS84.

    Returns
    -------
    list
        Köppen-Geiger climate codes.
    """

    if len(lons) != len(lats):
        raise ValueError("lons and lats must have the same length.")

    coordinates = list(zip(lons, lats))

    with rasterio.open(RASTER_URL) as src:

        # Raster should already be WGS84 according to the dataset.
        # Still check it explicitly.
        print("CRS:", src.crs)
        print("Resolution:", src.res)

        # sample() expects (x, y) = (longitude, latitude)
        samples = src.sample(coordinates)

        zones = []

        for value in samples:
            class_id = int(value[0])

            # Handle NoData / ocean / invalid cells
            if src.nodata is not None and class_id == src.nodata:
                zones.append(None)

            elif class_id not in KOPPEN_CLASSES:
                zones.append(None)

            else:
                zones.append(KOPPEN_CLASSES[class_id])

    return zones

In [195]:
lons = data['xmin'].to_numpy().tolist()
lats = data['ymin'].to_numpy().tolist()

zones = get_koppen_zones(lons, lats)

print(zones)


CRS: EPSG:4326
Resolution: (0.008333333333333333, 0.008333333333333333)
['Cfa', 'Dfa', 'Cwb', 'Cwb', 'BSk', 'Dwa', 'Dwa', 'Cfa', 'Cfb', 'Cfa', 'Dfa', 'Dfa', 'Cfa', 'Cfb', 'Dfa', 'Dwa', 'Cfb', 'Cfb', 'Cfa', 'BSh', 'BSk']


In [196]:
data['Climate'] = zones

In [197]:
data

,ID,xmin,ymin,Duration,Pop. Start,Climate
0,1,-84.37347649,33.75400957,3,"483,450",Cfa
1,3,-87.7126679,41.91046952,2,"2,718,782",Dfa
2,4,-99.20962876,19.43531808,1,"8,918,653",Cwb
3,5,-99.11203944,19.43137642,3,"8,920,000",Cwb
4,6,-111.9105146,40.74596957,6,"200,546",BSk
5,7,127.0737116,37.61597727,9,"2,660,000",Dwa
6,8,126.9129689,37.5548156,6,"2,660,000",Dwa
7,9,-74.00025074,40.7496983,8,"8,214,426",Cfa
8,10,2.253137216,48.91112189,11,"2,120,545",Cfb
9,11,-84.51188626,38.03708055,4,"322,600",Cfa


In [198]:
data['Match'] = None

In [199]:
target = 'BSk'

In [201]:
match_values = []
for index, row in data.iterrows():
    match_score = 0
    current_climate = row['Climate']

    if (current_climate[0] == target[0]):
        match_score += 4

    if (current_climate[1] == target[1]):
        match_score += 2

    if (current_climate[2] == target[2]):
        match_score += 1

    match_values.append(match_score)

In [202]:
match_values

[0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 7]

In [ ]:
# lsi = (0.25 * perimeter) / (sqrt(area))